# Linha — Evolução da População de MG (Interpolação/Extrapolação)

Dois gráficos de linha com a população de Minas Gerais ano a ano
(1970-2025), a partir da tabela já interpolada/extrapolada pelo método
geométrico (`interpolacao_extrapolacao.ipynb`) -- um ponto por ano,
acumulando todos os 853 municípios de MG.

Em ambos, o marcador distingue os anos com dado real do IBGE
(`ANOS_CENSITARIOS_POPULACAO` = 1970, 1980, 1991, 2000, 2010, 2022 --
maior e opaco) dos demais anos, estimados por interpolação (entre
censos) ou extrapolação (2023-2025) -- marcador menor e translúcido. A
linha em si é contínua: não há corte visual entre dado real e
estimado, só o marcador muda.

- **Gráfico 1** -- população total (soma das 8 faixas etárias e de
  todos os municípios), uma única linha.
- **Gráfico 2** -- a mesma soma por todos os municípios, mas uma linha
  por faixa etária (sem somar as faixas entre si).


In [ ]:
import sys
from pathlib import Path

# Garante que o notebook encontre os módulos locais (deste notebook e de utilitarios_comuns.py)
sys.path.append(str(Path.cwd()))

from caminho_utilitarios import preparar_sys_path
preparar_sys_path()


In [2]:
import pandas as pd

from utilitarios_comuns import (
    CAMINHO_POPULACAO,
    FAIXAS_ETARIAS_PADRAO,
    ANOS_CENSITARIOS_POPULACAO,
    registrar_log,
)
from utilitarios_graficos_populacionais import (
    agregar_populacao_estadual_por_ano,
    gerar_marcadores_categoria,
    montar_grafico_linha_populacao_com_destaque_censitario,
    montar_paineis_populacao_total_e_por_faixa,
    montar_grafico_barras_populacao_total,
    salvar_figura,
)


### Configuração

In [ ]:
PASTA_SAIDA_GRAFICOS = Path.cwd() / "Gráficos" / "Linha População"

SALVAR_FIGURAS = False  # True grava PNG/SVG/HTML de cada figura em PASTA_SAIDA_GRAFICOS

ANO_INICIAL_EXIBIDO = 1970
ANO_FINAL_EXIBIDO = 2025
LISTA_ANOS_COMPLETOS = list(range(ANO_INICIAL_EXIBIDO, ANO_FINAL_EXIBIDO + 1))

LARGURA_FIGURA = 1200
ALTURA_FIGURA = 700


### Carregamento e Agregação

Lê a tabela por município já interpolada/extrapolada e soma, para cada
ano, o TOTAL e cada faixa etária em todos os municípios de MG
(`agregar_populacao_estadual_por_ano`) -- nenhum valor é recalculado,
só somado.

In [4]:
tabela_populacao_municipios = pd.read_csv(CAMINHO_POPULACAO)
registrar_log(f"Tabela de população carregada: {tabela_populacao_municipios.shape[0]} municípios.")

serie_populacao_total, tabela_populacao_por_faixa = agregar_populacao_estadual_por_ano(
    tabela_populacao_municipios, FAIXAS_ETARIAS_PADRAO, LISTA_ANOS_COMPLETOS,
)

tabela_populacao_por_faixa.head()


[LOG] Tabela de população carregada: 853 municípios.


,0-9,10-19,20-29,30-39,40-49,50-59,60-69,70 OU MAIS
ANO,,,,,,,,
1970,1686715.0,1459944.0,911235.0,650135.0,467737.0,314208.0,180088.0,102544.0
1971,1681349.0,1471118.0,931157.0,696351.0,477262.0,321621.0,186558.0,107134.0
1972,1677330.0,1483308.0,952552.0,747067.0,487252.0,329414.0,193392.0,112066.0
1973,1674641.0,1496648.0,975483.0,802856.0,497726.0,337531.0,200564.0,117296.0
1974,1673391.0,1511138.0,1000102.0,864319.0,508749.0,346041.0,208110.0,122854.0


### Gráficos 1 e 2 — População Total e por Faixa Etária (painéis A/B)

Os dois gráficos (população total acumulada e população por faixa etária) ficam na mesma imagem, como painéis A) e B), com fonte, altura e espaçamento padronizados e eixo X compartilhado. Os anos censitários (dado real do IBGE) são destacados com marcador maior, rótulo em negrito e uma linha vertical pontilhada atravessando os dois painéis.

In [ ]:
series_por_faixa_tabela = tabela_populacao_por_faixa[list(FAIXAS_ETARIAS_PADRAO)]
marcadores_por_faixa = gerar_marcadores_categoria(list(FAIXAS_ETARIAS_PADRAO))

# Monta a figura com os dois painéis (A: total, B: por faixa etária) empilhados
figura_populacao_paineis = montar_paineis_populacao_total_e_por_faixa(
    serie_total=serie_populacao_total,
    tabela_por_faixa=series_por_faixa_tabela,
    anos_censitarios=ANOS_CENSITARIOS_POPULACAO,
    ordem_categorias=list(FAIXAS_ETARIAS_PADRAO),
    marcadores_categorias=marcadores_por_faixa,
    titulo_eixo_x="Ano", titulo_eixo_y="População",
    largura=LARGURA_FIGURA, altura=ALTURA_FIGURA * 2,
)
figura_populacao_paineis.show()

if SALVAR_FIGURAS:
    salvar_figura(figura_populacao_paineis, "populacao_total_e_por_faixa_etaria_mg", PASTA_SAIDA_GRAFICOS)

### Gráfico 3 — População Total por Ano, em barras (mesma soma do Gráfico 1)

In [6]:
figura_populacao_total_barras = montar_grafico_barras_populacao_total(
    serie_total=serie_populacao_total,
    anos_censitarios=ANOS_CENSITARIOS_POPULACAO,
    titulo="Evolução da População de Minas Gerais (1970-2025)",
    titulo_eixo_x="Ano", titulo_eixo_y="População",
    largura=LARGURA_FIGURA, altura=ALTURA_FIGURA,
)
figura_populacao_total_barras.show()

if SALVAR_FIGURAS:
    salvar_figura(figura_populacao_total_barras, "populacao_total_mg_barras", PASTA_SAIDA_GRAFICOS)